t-Testing

In [41]:
import pandas as pd
import numpy as np
from scipy import stats

Load the Data

In [42]:
df = pd.read_csv('../../output/processed_data/05_filtered.csv')

Daily Abnormal Returns

In [43]:
# AR was computed and sanity-checked in NB05.
# It is loaded directly from 05_filtered.csv — no need to recompute here.
assert 'AR' in df.columns, \
    "AR column missing from 05_filtered.csv — re-run NB05 first."
print(f'✓ AR loaded from 05_filtered.csv  ({df["AR"].notna().sum()} non-null values)')

✓ AR loaded from 05_filtered.csv  (69390 non-null values)


Event Window

In [44]:
windows = {
    'CAR(-5,-1)': (-5, -1),
    'CAR(0,0)': (0, 0),
    'CAR(0,+5)': (0, 5),
    'CAR(-5,+30)': (-5, 30)
}

t-test Function

In [45]:
def run_cross_sectional_ttest(data, start_day, end_day, window_name):
    #Filter the data
    window_data = data[(data['day'] >= start_day) & (data['day'] <= end_day)]

    #sum daily ARto get CAR
    stock_cars = window_data.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

    results = []

    # Sector level t-test
    sectors = stock_cars['sector'].unique()

    for sector in sectors:
        sector_data = stock_cars[stock_cars['sector'] == sector]['CAR']
        n_stocks = len(sector_data)
        
        if n_stocks > 1:
            mean_car = sector_data.mean()
            std_car = sector_data.std()
            t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
        else:
            mean_car = sector_data.mean()
            std_car = np.nan
            t_stat = np.nan
            p_val = np.nan

        results.append({
            'Window': window_name,
            'Level': 'Sector',
            'Name': sector,
            'N': n_stocks,
            'Mean_CAR(%)': mean_car * 100, # Convert to percentage for readability
            'Std_Dev(%)': std_car * 100,
            't-statistic': t_stat,
            'p-value': p_val,
            'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
        })

    #Market t-test
    all_cars = stock_cars['CAR']
    n_total = len(all_cars)
    t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_cars, popmean=0)

    results.append({
        'Window': window_name,
        'Level': 'Market',
        'Name': 'ALL STOCKS',
        'N': n_total,
        'Mean_CAR(%)': all_cars.mean() * 100, 
        'Std_Dev(%)': all_cars.std() * 100,
        't-statistic': t_stat_mkt,
        'p-value': p_val_mkt,
        'Significant_at_5%': p_val_mkt < 0.05 
    })

    return pd.DataFrame(results)



Test all windows

In [46]:
all_results = []

for window_name, (start, end) in windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    all_results.append(window_results_df)

final_ttest_results = pd.concat(all_results,ignore_index=True)

formatted_results = final_ttest_results.copy()
formatted_results['Mean_CAR(%)'] = formatted_results['Mean_CAR(%)'].round(2)
formatted_results['Std_Dev(%)'] = formatted_results['Std_Dev(%)'].round(2)
formatted_results['t-statistic'] = formatted_results['t-statistic'].round(3)
formatted_results['p-value'] = formatted_results['p-value'].round(4)

Results

In [47]:
display_df = formatted_results[formatted_results['Window'] == 'CAR(-5,+30)'].sort_values(by='Mean_CAR(%)')
print(display_df[['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']])


formatted_results.to_csv('../../output/tables/ttest_results.csv', index=False)

                          Name    N  Mean_CAR(%)  t-statistic  p-value  \
81              Transportation    3       -24.69       -1.690   0.2330   
85                      Energy    3       -17.93       -1.187   0.3571   
76                   Retailing   14       -16.58       -0.779   0.4498   
67                   Insurance   13       -13.09       -4.618   0.0006   
77                      Others    1       -13.08          NaN      NaN   
84                   Utilities    9       -11.67       -2.700   0.0271   
73           Consumer Services   33       -10.19       -6.826   0.0000   
66      Diversified Financials   39        -9.55       -4.436   0.0001   
87                  ALL STOCKS  279        -5.86       -4.083   0.0001   
71                   Materials   22        -4.90       -0.688   0.4989   
75                 Real Estate   17        -4.69       -1.181   0.2549   
78          Household Products    1        -4.36          NaN      NaN   
80              Food Retailing    4   